# Zaid Hybrid IDS — Step 4: Baseline Detector Models
This notebook trains three fixed baseline detectors on the training partition and evaluates them on the validation partition. The test partition is deliberately not loaded or used. The task is binary detection: Benign = 0 and either brute-force class = 1.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from time import perf_counter
import gc
import json
import platform
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
SPLIT_DIR = PROJECT_DIR / 'data' / 'splits'
RESULT_DIR = PROJECT_DIR / 'results'
MODEL_DIR = PROJECT_DIR / 'models'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLIT_DIR / 'train.parquet'
VALIDATION_PATH = SPLIT_DIR / 'validation.parquet'
FEATURE_MANIFEST_PATH = RESULT_DIR / 'feature_sets_manifest.json'
SEED = 20260829

assert TRAIN_PATH.exists()
assert VALIDATION_PATH.exists()
assert FEATURE_MANIFEST_PATH.exists()

print('Loading training and validation only...')
train_df = pd.read_parquet(TRAIN_PATH)
validation_df = pd.read_parquet(VALIDATION_PATH)
with open(FEATURE_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    feature_manifest = json.load(file_handle)

assert len(train_df) == 731315
assert len(validation_df) == 156714
print('Train:', train_df.shape)
print('Validation:', validation_df.shape)
print('Test partition loaded: False')

Loading training and validation only...
Train: (731315, 81)
Validation: (156714, 81)
Test partition loaded: False


In [3]:
FEATURES = feature_manifest['full_feature_set']
assert len(FEATURES) == 61
assert 'Dst Port' in FEATURES

X_train = train_df[FEATURES].astype(np.float32)
X_validation = validation_df[FEATURES].astype(np.float32)
y_train_original = train_df['Label'].astype(str).reset_index(drop=True)
y_validation_original = validation_df['Label'].astype(str).reset_index(drop=True)
y_train = y_train_original.ne('Benign').astype(np.int8).to_numpy()
y_validation = y_validation_original.ne('Benign').astype(np.int8).to_numpy()

assert np.isfinite(X_train.to_numpy(copy=False)).all()
assert np.isfinite(X_validation.to_numpy(copy=False)).all()

print('Feature count:', len(FEATURES))
print('Train binary labels:', dict(zip(*np.unique(y_train, return_counts=True))))
print('Validation binary labels:', dict(zip(*np.unique(y_validation, return_counts=True))))
print('Data preparation: PASSED')

Feature count: 61
Train binary labels: {np.int8(0): np.int64(464662), np.int8(1): np.int64(266653)}
Validation binary labels: {np.int8(0): np.int64(99570), np.int8(1): np.int64(57144)}
Data preparation: PASSED


In [4]:
results = []
validation_predictions = pd.DataFrame({
    'validation_row': np.arange(len(validation_df), dtype=np.int64),
    'flow_group_id': validation_df['flow_group_id'].to_numpy(),
    'original_label': y_validation_original,
    'binary_label': y_validation
})

def fit_and_evaluate(model_name, model):
    print(f'\nTraining {model_name}...')
    fit_start = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - fit_start

    inference_start = perf_counter()
    attack_probability = model.predict_proba(X_validation)[:, 1]
    inference_seconds = perf_counter() - inference_start
    prediction = (attack_probability >= 0.5).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_validation, prediction, labels=[0, 1]
    ).ravel()
    false_positive_rate = fp / (fp + tn) if (fp + tn) else 0.0

    ftp_mask = y_validation_original.eq('FTP-BruteForce').to_numpy()
    ssh_mask = y_validation_original.eq('SSH-Bruteforce').to_numpy()
    ftp_attack_recall = float(prediction[ftp_mask].mean())
    ssh_attack_recall = float(prediction[ssh_mask].mean())

    model_path = MODEL_DIR / f'{model_name}.joblib'
    joblib.dump(model, model_path, compress=3)
    model_size_mb = model_path.stat().st_size / (1024 ** 2)

    row = {
        'model': model_name,
        'threshold': 0.5,
        'accuracy': accuracy_score(y_validation, prediction),
        'balanced_accuracy': balanced_accuracy_score(y_validation, prediction),
        'precision': precision_score(y_validation, prediction, zero_division=0),
        'recall': recall_score(y_validation, prediction, zero_division=0),
        'f1': f1_score(y_validation, prediction, zero_division=0),
        'macro_f1': f1_score(y_validation, prediction, average='macro'),
        'pr_auc': average_precision_score(y_validation, attack_probability),
        'roc_auc': roc_auc_score(y_validation, attack_probability),
        'brier_score': brier_score_loss(y_validation, attack_probability),
        'false_positive_rate': false_positive_rate,
        'ftp_attack_recall': ftp_attack_recall,
        'ssh_attack_recall': ssh_attack_recall,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'fit_seconds': fit_seconds,
        'validation_inference_seconds': inference_seconds,
        'batch_mean_microseconds_per_flow': (inference_seconds / len(y_validation)) * 1e6,
        'batch_throughput_flows_per_second': len(y_validation) / inference_seconds,
        'model_size_mb': model_size_mb
    }
    results.append(row)
    validation_predictions[f'{model_name}_probability'] = attack_probability.astype(np.float32)
    validation_predictions[f'{model_name}_prediction'] = prediction

    print(pd.Series(row).to_string())
    print('Saved model:', model_path)
    gc.collect()
    return model

print('Evaluation function: READY')

Evaluation function: READY


In [5]:
linear_continuous_features = [feature for feature in FEATURES if feature != 'Protocol']
linear_preprocessor = ColumnTransformer(
    transformers=[
        ('continuous', StandardScaler(), linear_continuous_features),
        ('protocol', OneHotEncoder(
            handle_unknown='ignore', sparse_output=False, dtype=np.float32
        ), ['Protocol'])
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
sgd_logistic = Pipeline([
    ('preprocess', linear_preprocessor),
    ('classifier', SGDClassifier(
        loss='log_loss', penalty='l2', alpha=1e-4, max_iter=2000,
        tol=1e-4, class_weight='balanced', average=True,
        random_state=SEED, n_jobs=-1
    ))
])
sgd_logistic = fit_and_evaluate('sgd_logistic', sgd_logistic)


Training sgd_logistic...
model                                 sgd_logistic
threshold                                      0.5
accuracy                                  0.999872
balanced_accuracy                         0.999888
precision                                 0.999703
recall                                    0.999948
f1                                        0.999825
macro_f1                                  0.999862
pr_auc                                    0.999969
roc_auc                                   0.999988
brier_score                                0.00014
false_positive_rate                       0.000171
ftp_attack_recall                              1.0
ssh_attack_recall                         0.999893
tn                                           99553
fp                                              17
fn                                               3
tp                                           57141
fit_seconds                               7.762803
valid

In [6]:
random_forest = RandomForestClassifier(
    n_estimators=150, max_depth=20, min_samples_leaf=2,
    max_features='sqrt', class_weight='balanced_subsample',
    n_jobs=-1, random_state=SEED
)
random_forest = fit_and_evaluate('random_forest', random_forest)


Training random_forest...
model                                random_forest
threshold                                      0.5
accuracy                                  0.999994
balanced_accuracy                         0.999995
precision                                 0.999983
recall                                         1.0
f1                                        0.999991
macro_f1                                  0.999993
pr_auc                                         1.0
roc_auc                                        1.0
brier_score                               0.000011
false_positive_rate                        0.00001
ftp_attack_recall                              1.0
ssh_attack_recall                              1.0
tn                                           99569
fp                                               1
fn                                               0
tp                                           57144
fit_seconds                             161.414875
vali

In [7]:
hist_gradient_boosting = HistGradientBoostingClassifier(
    learning_rate=0.08, max_iter=200, max_leaf_nodes=31,
    min_samples_leaf=20, l2_regularization=1.0,
    class_weight='balanced', early_stopping=False,
    random_state=SEED
)
hist_gradient_boosting = fit_and_evaluate(
    'hist_gradient_boosting', hist_gradient_boosting
)


Training hist_gradient_boosting...
model                                hist_gradient_boosting
threshold                                               0.5
accuracy                                           0.999994
balanced_accuracy                                  0.999995
precision                                          0.999983
recall                                                  1.0
f1                                                 0.999991
macro_f1                                           0.999993
pr_auc                                             0.999978
roc_auc                                            0.999994
brier_score                                        0.000006
false_positive_rate                                 0.00001
ftp_attack_recall                                       1.0
ssh_attack_recall                                       1.0
tn                                                    99569
fp                                                        1
fn  

In [8]:
assert len(results) == 3, 'Run all three model cells before this cell'
metrics_df = pd.DataFrame(results).sort_values(
    ['macro_f1', 'recall', 'false_positive_rate'],
    ascending=[False, False, True]
).reset_index(drop=True)

metrics_path = RESULT_DIR / 'baseline_validation_metrics.csv'
predictions_path = RESULT_DIR / 'baseline_validation_predictions.parquet'
metrics_df.to_csv(metrics_path, index=False)
validation_predictions.to_parquet(predictions_path, index=False, compression='zstd')

preliminary_validation_winner = str(metrics_df.iloc[0]['model'])
run_manifest = {
    'stage': 'fixed_baseline_validation',
    'seed': SEED,
    'task': 'binary Benign versus Attack',
    'feature_set': 'full_feature_set',
    'feature_count': len(FEATURES),
    'models': ['sgd_logistic', 'random_forest', 'hist_gradient_boosting'],
    'selection_partition': 'validation',
    'test_partition_loaded_or_used': False,
    'fixed_threshold': 0.5,
    'primary_ranking_metric': 'macro_f1',
    'preliminary_validation_winner': preliminary_validation_winner,
    'latency_note': 'Reported latency is mean batched validation inference time per flow; real-time per-flow latency is measured later',
    'sklearn_version': sklearn.__version__,
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'python_version': sys.version,
    'platform': platform.platform()
}
manifest_path = RESULT_DIR / 'baseline_run_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as file_handle:
    json.dump(run_manifest, file_handle, ensure_ascii=False, indent=2)

display_columns = [
    'model', 'macro_f1', 'f1', 'precision', 'recall', 'pr_auc',
    'false_positive_rate', 'ftp_attack_recall', 'ssh_attack_recall',
    'batch_mean_microseconds_per_flow', 'model_size_mb'
]
print(metrics_df[display_columns].to_string(index=False))
print('\nPreliminary validation winner:', preliminary_validation_winner)
print('Saved:', metrics_path)
print('Saved:', predictions_path)
print('Saved:', manifest_path)
print('BASELINE VALIDATION PASSED: test partition remains untouched.')

                 model  macro_f1       f1  precision   recall   pr_auc  false_positive_rate  ftp_attack_recall  ssh_attack_recall  batch_mean_microseconds_per_flow  model_size_mb
         random_forest  0.999993 0.999991   0.999983 1.000000 1.000000             0.000010                1.0           1.000000                          6.882657       0.304957
hist_gradient_boosting  0.999993 0.999991   0.999983 1.000000 0.999978             0.000010                1.0           1.000000                         12.108243       0.039079
          sgd_logistic  0.999862 0.999825   0.999703 0.999948 0.999969             0.000171                1.0           0.999893                          1.217088       0.004158

Preliminary validation winner: random_forest
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/baseline_validation_metrics.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/baseline_validation_predictions.parquet
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/base